In [ ]:
# Cell 1: Imports and config
import os
import json
import yaml
from itertools import product
from src.utils.data_preprocessing import deduplicate_session_notes
from src.tools.ollama_tool import OllamaTool

# Cell 2: Parameter sweep
learning_rates = [1e-4, 2e-4, 5e-4]
epochs_list = [2, 3, 5]
lora_ranks = [8, 16, 32]
best_score = -float("inf")
best_config = None

# Load user notes
with open("data/sample_notes.json") as f:
    raw_notes = json.load(f)
cleaned_notes = deduplicate_session_notes(raw_notes)

# Cell 3: Evaluation function
def evaluate_model(base_model, ft_model, test_prompts):
    # Use Ollama to compare responses
    scores = []
    for prompt in test_prompts:
        base_resp = ollama.generate(base_model, prompt)
        ft_resp = ollama.generate(ft_model, prompt)
        # Simple BLEU/ROUGE; in production use an evaluator LLM
        scores.append(calculate_bleu(base_resp, ft_resp))
    return sum(scores)/len(scores)

# Cell 4: Sweep loop
tool = OllamaTool()
for lr, ep, lr_rank in product(learning_rates, epochs_list, lora_ranks):
    ft_model = f"llama3.2:3b-ft-lr{lr}-ep{ep}-rank{lr_rank}"
    # Simulate fine-tuning (dry run)
    status = tool._run("fine_tuning_dry_run", user_notes=cleaned_notes, model_name=ft_model)
    if status.get("ready"):
        # Trigger real fine-tuning (would call Ollama API)
        print(f"Training {ft_model}...")
        # After training, evaluate
        score = evaluate_model("llama3.2:3b", ft_model, ["Tell me about yourself", "What are your strengths?"])
        if score > best_score:
            best_score = score
            best_config = {"learning_rate": lr, "epochs": ep, "lora_rank": lr_rank}

# Cell 5: Promote best config
if best_config:
    with open("config/llm_config.yaml") as f:
        llm_cfg = yaml.safe_load(f)
    llm_cfg["fine_tuning"] = best_config
    with open("config/llm_config.yaml", "w") as f:
        yaml.dump(llm_cfg, f)
    print(f"Promoted best config: {best_config}")

ollama_fine_tuning_experiment.ipynb